# LSTM vs. Transformer for music generation

Run the six experiments and temperature samples. Choose **Runtime > Change runtime type > GPU** first. The recorded results in the repository were run locally; this notebook is a Colab entry point. A different GPU or package version may change timings and results.

## Upload the project

Upload `lstm-transformer-music.zip` (or the ZIP downloaded from GitHub). This extracts into a new folder so repeated runs do not overwrite earlier outputs.

In [ ]:
from google.colab import files
from pathlib import Path
import tempfile
import zipfile
import os

uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
assert len(archives) == 1, "Upload one project ZIP."
folder = Path(tempfile.mkdtemp(prefix="music_project_", dir="/content"))
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        target = (folder / member.filename).resolve()
        if not target.is_relative_to(folder.resolve()):
            raise ValueError("Unsafe archive path")
    archive.extractall(folder)
projects = list(folder.rglob("run_experiments.py"))
assert len(projects) == 1, "Expected one music project."
os.chdir(projects[0].parent)
print(Path.cwd())

In [ ]:
%pip install -r requirements.txt

## Check and train

The checks cover causal masking, shifted targets and data separation. Training compares two architectures, two widths and two sequence lengths. Each run uses the same number of training characters; shorter sequences require more optimizer updates.

In [ ]:
import subprocess
import sys

def run(script):
    subprocess.run([sys.executable, script], check=True)

run("test_models.py")
run("run_experiments.py")

## Generate samples and plots

MIDI export may fail for malformed ABC notation. Failed examples are kept. An exported MIDI is not a measure of musical quality.

In [ ]:
run("sample.py")
run("plot_results.py")
from IPython.display import Image, display
display(Image(filename="runs/figures/comparison.png"))
display(Image(filename="runs/figures/temperature.png"))

## Save this run

Download metrics, plots, generated samples and model checkpoints before closing the runtime.

In [ ]:
import shutil
archive_path = shutil.make_archive("music_experiment_results", "zip", "runs")
files.download(archive_path)